# Checking if `check_cartesian_format` works

## Intializing

In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

import pandas as pd
from notebooks.radp_library import (preprocess_ue_data)
from notebooks.radp_library import get_ue_data
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO

In [ ]:
# Make sure to place it in the mobility_robustness_optimization.py file

def _check_cartesian_format(self, df: pd.DataFrame) -> bool:
    # Get the expected cell_ids from the topology
    expected_cell_ids = set(str(cid) for cid in self.topology["cell_id"].unique())

    # Make a copy of the DataFrame and ensure cell_id is in string format for consistency
    df = df.copy()
    df["cell_id"] = df["cell_id"].astype(str)

    # Drop exact duplicates — these are legal
    df_dedup = df.drop_duplicates(subset=["longitude", "latitude", "cell_id", "cell_rxpwr_dbm"])

    # Iterate over the groups of data by unique (longitude, latitude)
    for (lon, lat), group in df_dedup.groupby(["longitude", "latitude"]):
        # Get the (cell_id, rxpwr) pairs for the group
        cell_rxpwr_pairs = set(zip(group["cell_id"], group["cell_rxpwr_dbm"]))

        # Get the full (cell_id, rxpwr) pairs from the subset of data at that location
        subset = df[(df["longitude"] == lon) & (df["latitude"] == lat)]
        full_combos = set(zip(subset["cell_id"], subset["cell_rxpwr_dbm"]))

        # Check if there is any inconsistency between the unique pairs in the group and the full set
        if cell_rxpwr_pairs != full_combos:
            raise ValueError(
                f"Inconsistent (cell_id, rxpwr) attachments for location ({lon}, {lat})."
            )

        # Get the present cell_ids from the cell_rxpwr_pairs
        cell_ids_present = {cid for cid, _ in cell_rxpwr_pairs}

        # Calculate the missing cell_ids
        missing = expected_cell_ids - cell_ids_present

        # If there are any missing or extra cell_ids, raise a ValueError
        if missing:
            raise ValueError(
                f"Location ({lon}, {lat}) has missing cell_ids {missing}"
                f"(expected: {expected_cell_ids}, found: {cell_ids_present})"
            )

    return True


In [ ]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 2,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 1,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 0,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 0,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 0,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [ ]:
topology = pd.read_csv('data/sim_data/topology.csv')
topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

topology

In [ ]:
ue_data = get_ue_data(params)
ue_data = ue_data.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
ue_data.drop(columns={'tick', 'mock_ue_id'}, inplace=True)
ue_data

In [ ]:
master_data = preprocess_ue_data(ue_data, topology)
master_data

## **Scenario 1:** Data is in cartesian form **[Should Pass]**

In [ ]:
input_data = master_data.copy()
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(input_data)

## **Senario 2:** There are extra or fewer data **[Should Fail]**

In [ ]:
scenario_2 = master_data[['longitude', 'latitude', 'cell_id', 'cell_rxpwr_dbm']].copy()

extra_ue = pd.DataFrame({
    "longitude": [129.091599],
    "latitude": [35.525018],
    "cell_id": [4.0],
    "cell_rxpwr_dbm": [-100.219108],
})

# Duplicate UE
duplicate_row = scenario_2.iloc[0].copy() 
scenario_2_extra = pd.concat([scenario_2, duplicate_row.to_frame().T], ignore_index=True)

# Extra gibberish  
# scenario_2_extra = pd.concat([scenario_2, extra_ue], ignore_index=True)

scenario_2_fewer = scenario_2.drop(0).copy()

scenario_2_extra = scenario_2_extra.sort_values(by=['longitude', 'latitude']).copy()

### Extra row [this should pass] --> [if fails, fine]

In [ ]:
scenario_2_extra

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_2_extra)

### Extra row [this should fail]

In [ ]:
scenario_2_extra.loc[6] = [1.0, 2.0, 1.0, -100.219108]
scenario_2_extra

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_2_extra)

### Extra row [this should fail]

In [ ]:
scenario_2_extra.loc[6] = [1.0, 2.0, 4.0, -100.219108]
scenario_2_extra

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_2_extra)

### Fewer row

In [ ]:
scenario_2_fewer

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_2_fewer)

## **Scenario 3:** Same dimensions as `master` but duplicate or gibberish row **[Should Fail]**

### Duplicate row

In [ ]:
scenario_3 = master_data[['longitude', 'latitude', 'cell_id', 'cell_rxpwr_dbm']].copy()


scenario_3_duplicate = scenario_3.drop(1).copy()
duplicate_row = scenario_3.iloc[0].copy() 
scenario_3_duplicate = pd.concat([scenario_3_duplicate, duplicate_row.to_frame().T], ignore_index=True)
scenario_3_duplicate.sort_values(by=['longitude', 'cell_id'], inplace=True)


scenario_3_duplicate

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_3_duplicate)

### Gibberish Row

In [ ]:
scenario_3_gibberish = scenario_3.drop(2).copy()

extra_ue = pd.DataFrame({
    "longitude": [129.091599],
    "latitude": [35.525018],
    "cell_id": [4.0],
    "cell_rxpwr_dbm": [-100.219108],
})

scenario_3_gibberish = pd.concat([scenario_3_gibberish, extra_ue], ignore_index=True)
scenario_3_gibberish.sort_values(by=['longitude', 'cell_id'], inplace=True)

scenario_3_gibberish

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_3_gibberish)

### `Lat_1, Lon_1` missing a row **&** `Lat_2, Lon_2` has an extra gibberish row **[should fail]

In [ ]:
scenario_3_missing_and_gibberish = scenario_3.copy()

extra_ue = pd.DataFrame({
    "longitude": [129.091599],
    "latitude": [35.525018],
    "cell_id": [4.0],
    "cell_rxpwr_dbm": [-100.219108],
})

scenario_3_missing_and_gibberish = pd.concat([scenario_3_missing_and_gibberish, extra_ue], ignore_index=True)
scenario_3_missing_and_gibberish.sort_values(by=['longitude', 'cell_id'], inplace=True)
scenario_3_missing_and_gibberish.drop(5, inplace=True)

scenario_3_missing_and_gibberish

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_3_missing_and_gibberish)

## **Scenario 4**: UE returns to same `Lat, Lon` **[Should Pass]**

In [ ]:
scenario_4 = master_data[['longitude', 'latitude', 'cell_id', 'cell_rxpwr_dbm']].copy()


duplicate_row_1 = scenario_4.iloc[0].copy() 
duplicate_row_2 = scenario_4.iloc[1].copy() 
duplicate_row_3 = scenario_4.iloc[2].copy() 
scenario_4 = pd.concat([scenario_4, duplicate_row_1.to_frame().T, duplicate_row_2.to_frame().T, duplicate_row_3.to_frame().T], ignore_index=True)
scenario_4.sort_values(by=['longitude', 'cell_id'], inplace=True)

scenario_4

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_4)

## **Scenario 5**: UE returns to same `Lat, Lon` but attches to the wrong cell_id(s) **[Should Fail]**

In [ ]:
scenario_5 = master_data[['longitude', 'latitude', 'cell_id', 'cell_rxpwr_dbm']].copy()


duplicate_row_1 = scenario_5.iloc[0].copy() 
duplicate_row_2 = scenario_5.iloc[1].copy() 
duplicate_row_3 = scenario_5.iloc[2].copy() 
scenario_5 = pd.concat([scenario_5, duplicate_row_1.to_frame().T, duplicate_row_2.to_frame().T, duplicate_row_2.to_frame().T], ignore_index=True)
scenario_5.sort_values(by=['longitude', 'cell_id'], inplace=True)

scenario_5

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_5)

In [ ]:
scenario_5.loc[8, 'cell_id'] = 4
scenario_5

In [ ]:
mro = SimpleMRO(params, topology)
mro.train_or_update_rf_twin(scenario_5)